In [0]:

# Phase 4 - Step 2: Read Silver Customer Data

silver_table = "workspace.customer_data_pipeline.silver_customers"

df_silver = spark.table(silver_table)

print("Total Silver records:", df_silver.count())

display(df_silver.orderBy("customer_id"))


In [0]:

# Phase 4 - Step 3: Calculate Customer Counts by City

from pyspark.sql import functions as F

df_gold_city = (
    df_silver
    .groupBy("city")
    .agg(
        F.countDistinct("customer_id").alias("total_customers")
    )
    .orderBy("city")
)

display(df_gold_city)


In [0]:

# Phase 4 - Step 4: Validate Gold Customer Counts

# Total customers in Silver
silver_count = df_silver.select("customer_id").distinct().count()

# Total customers reported in Gold
gold_count = df_gold_city.agg(
    F.sum("total_customers")
).first()[0]

# Number of city groups
city_count = df_gold_city.count()

print("Silver unique customers:", silver_count)
print("Gold total customers:", gold_count)
print("Total city groups:", city_count)

# Data quality assertions
assert silver_count == 4, "Unexpected Silver customer count"
assert gold_count == silver_count, "Gold and Silver counts do not match"
assert city_count == 4, "Unexpected number of city groups"

print("GOLD DATA QUALITY VALIDATION PASSED")


In [0]:

# Phase 4 - Step 5: Save Gold Customer Analytics

gold_table = "workspace.customer_data_pipeline.gold_customers_by_city"

(
    df_gold_city.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print("Gold Delta table created successfully!")
print("Table:", gold_table)


In [0]:

# Phase 4 - Step 6: Verify Saved Gold Delta Table

gold_table = "workspace.customer_data_pipeline.gold_customers_by_city"

# Read the saved Gold table
df_verify_gold = spark.table(gold_table)

# Validate saved records
city_count = df_verify_gold.count()

total_customers = (
    df_verify_gold
    .agg(F.sum("total_customers"))
    .first()[0]
)

print("Saved Gold city groups:", city_count)
print("Total customers in Gold:", total_customers)

# Data quality validation
assert city_count == 4, "Incorrect number of city groups"
assert total_customers == 4, "Gold customer count mismatch"

print("GOLD DELTA TABLE VALIDATION PASSED")

# Display saved Gold analytics
display(df_verify_gold.orderBy("city"))
